# BUỔI 05 — APACHE SPARK: XỬ LÝ TRONG BỘ NHỚ VÀ API PYSPARK

**Notebook thực hành dành cho học viên**

|  |  |
|:---|:---|
| **Bộ dữ liệu** | `wordcount_corpus.txt` · `transactions.csv` — **dùng lại nguyên vẹn từ Buổi 04** |
| **Thời lượng** | 60 phút (Lý thuyết 25' + Thực hành 35') |
| **Môi trường** | PySpark ≥ 3.5 chế độ `local[*]` + cụm Docker cho bước D8 |

---

## Buổi này nối tiếp Buổi 04 như thế nào

Buổi 04 kết thúc bằng một con số khó chịu: **job MapReduce xử lý 7 KB và job xử lý 18,5 MB đều mất khoảng 30 giây.** Thời gian gần như **không phụ thuộc vào lượng dữ liệu**.

Hôm nay ta chạy **đúng hai bài toán đó, trên đúng hai tệp đó**, bằng Apache Spark.

```text
BUỔI 04                                  BUỔI 05
Ghi trung gian xuống ĐĨA           ──►   Giữ trung gian trong BỘ NHỚ
Viết mapper.py + reducer.py        ──►   .flatMap().map().reduceByKey()
Mỗi job = một chương trình riêng   ──►   Nhiều phép biến đổi trong một phiên
Hỏng thì chạy lại từ dữ liệu đĩa   ──►   Dựng lại từ LINEAGE, không cần ghi đĩa
~30 giây / job                     ══►   KẾT QUẢ PHẢI GIỐNG HỆT, thời gian < 5 giây
```

> **Mục tiêu lớn nhất:** con số phải khớp **tuyệt đối** với Buổi 04 — **373** từ khác nhau, **1.144** tổng số từ, **206.744.802.000 VND** tổng doanh thu. Khi kết quả giống hệt mà thời gian giảm gần mười lần, bạn hiểu ngay: Spark **không tính khác**, Spark chỉ **tránh được việc ghi đĩa**.

---

## Cách làm việc với notebook này

Mỗi nhiệm vụ có **4 thành phần**: Nhiệm vụ → PROMPT CHO AI AGENT → GỢI Ý GIẢI → TỰ KIỂM TRA.

> **Trước khi bắt đầu:**
> ```
> pip install "pyspark>=3.5.1"
> java -version          # PySpark 4.x cần Java >= 17; PySpark 3.5.x cần Java 8/11/17
> ```
> **Bắt buộc phải có `lab4/outputs/lab4_timings.json`** — buổi này đọc trực tiếp tệp đó. Chưa có thì quay lại chạy Buổi 04 tới D8.
>
> D8 cần thêm cụm Docker:
> ```
> docker compose up -d hadoop-namenode hadoop-datanode spark-master spark-worker
> ```

> **CẢNH BÁO QUAN TRỌNG NHẤT CỦA BUỔI HỌC.** **Đừng** nối PySpark trên máy thật vào cụm Docker bằng `.master("spark://localhost:7077")` trừ khi phiên bản PySpark của bạn **trùng khớp** với Spark trong container (khóa học dùng **3.5.1**). Lệch phiên bản cho lỗi `java.io.InvalidClassException` hoặc treo mãi ở `Initial job has not accepted any resources`.
>
> Toàn bộ D1–D7 dùng `.master("local[*]")` — Spark chạy ngay trong tiến trình Python, mỗi lõi CPU là một "máy". D8 mới chạy Spark **bên trong container**.


---
## Ô THIẾT LẬP — CHẠY Ô NÀY ĐẦU TIÊN

Ô này dò đường dẫn dữ liệu, nạp thời gian chạy của Buổi 04, và kiểm tra Java/PySpark. **Không cần sửa gì.**


In [ ]:
# =============================================================================
# Ô THIẾT LẬP — chạy đầu tiên, không cần chỉnh sửa
# =============================================================================
import json
import subprocess
import sys
import time
from pathlib import Path


def _goc_du_an() -> Path:
    """Đi ngược cây thư mục để tìm gốc dự án."""
    here = Path.cwd().resolve()
    for tm in [here, *here.parents]:
        if (tm / "data" / "make_lab_datasets.py").exists():
            return tm
    raise FileNotFoundError("Không tìm thấy gốc dự án — hãy mở notebook từ trong thư mục dự án.")


GOC         = _goc_du_an()
CORPUS_PATH = GOC / "data" / "raw" / "wordcount_corpus.txt"
TX_PATH     = GOC / "data" / "raw" / "transactions.csv"
OUTPUT_DIR  = GOC / "lab5" / "outputs"
SPARK_JOBS  = GOC / "lab5" / "spark_jobs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not (CORPUS_PATH.exists() and TX_PATH.exists()):
    print("Chưa có dữ liệu — đang sinh bằng data/make_lab_datasets.py ...")
    subprocess.run([sys.executable, str(GOC / "data" / "make_lab_datasets.py")], check=True)

# --- Nạp thời gian chạy của Buổi 04 -------------------------------------------
_tep_b04 = GOC / "lab4" / "outputs" / "lab4_timings.json"
MAC_DINH_B04 = {"yarn_wordcount_s": 29.19, "yarn_revenue_s": 29.20,
                "python_ong_dan_unix_s": 5.92, "pandas_s": 0.32}
if _tep_b04.exists():
    TIMINGS_B04 = json.loads(_tep_b04.read_text(encoding="utf-8"))
    _nguon_b04 = "đọc từ lab4/outputs/lab4_timings.json"
else:
    TIMINGS_B04 = MAC_DINH_B04
    _nguon_b04 = "MẶC ĐỊNH (đo trên máy soạn bài) — hãy chạy Buổi 04 để có số của chính bạn"

# --- Kết quả chuẩn của Buổi 04, dùng để đối chiếu ------------------------------
CHUAN_WC = {"so_tu_khac_nhau": 373, "tong_so_tu": 1144,
            "top": {"một": 43, "liệu": 27, "dữ": 27, "của": 26}}
CHUAN_REVENUE = {
    "Gia dụng":   (32231, 58925123000),
    "Điện tử":    (36296, 57872658000),
    "Thời trang": (27778, 47277119000),
    "Thực phẩm":  (59856, 18555159000),
    "Mỹ phẩm":    (19839, 17421590000),
    "Sách":       (24000,  6693153000),
}
TONG_DOANH_THU = 206_744_802_000

# --- Quy tắc chuẩn hóa văn bản — PHẢI GIỐNG HỆT Buổi 04 ------------------------
DAU_CAU  = str.maketrans({c: " " for c in '.,;:!?"()[]'})
HA_ASCII = str.maketrans("ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz")

print("=" * 78)
print("MÔI TRƯỜNG THỰC HÀNH — BUỔI 05 (APACHE SPARK · PYSPARK)")
print("=" * 78)
print(f"Gốc dự án        : {GOC}")
print(f"Kho văn bản      : {CORPUS_PATH.name:<24}{CORPUS_PATH.stat().st_size/1024:>10,.1f} KB")
print(f"Nhật ký giao dịch: {TX_PATH.name:<24}{TX_PATH.stat().st_size/1024**2:>10,.2f} MB")
print("-" * 78)

try:
    import pyspark
    print("PySpark          :", pyspark.__version__)
except ImportError:
    print("CHƯA CÀI PYSPARK. Chạy:  pip install \"pyspark>=3.5.1\"")

_java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java             :", (_java.stderr or _java.stdout).split("\n")[0] or "KHÔNG TÌM THẤY JAVA")
print("Python kernel    :", sys.executable)
print("-" * 78)
print("Thời gian Buổi 04 :", _nguon_b04)
for k in ("yarn_wordcount_s", "yarn_revenue_s"):
    if k in TIMINGS_B04:
        print(f"   {k:<26}{TIMINGS_B04[k]:>8.2f} s")
print("=" * 78)
print("Sẵn sàng. Chuyển sang D1.")


---
---
# D1. KHỞI TẠO SPARKSESSION
### 4 phút

### Nhiệm vụ

Tạo `SparkSession`, quan sát cấu hình, và mở Spark UI.

### Kết quả mong đợi

```text
Phiên bản Spark     : 4.2.0  (hoặc 3.5.x)
Chế độ chạy         : local[*]
Số lõi khả dụng     : 12     (bằng số lõi CPU máy bạn)
Thời gian khởi tạo  : khoảng 4 giây
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Khởi tạo SparkSession cho bài thực hành PySpark:
1. import SparkSession và functions as F từ pyspark.sql
2. Tạo SparkSession với:
   - appName "Buoi05_Spark"
   - master "local[*]"
   - config spark.sql.shuffle.partitions = 8
   - config spark.ui.showConsoleProgress = false
   Đo thời gian khởi tạo bằng time.perf_counter().
3. setLogLevel("ERROR") để bớt log rác
4. Gán sc = spark.sparkContext
5. In: phiên bản Spark, sc.master, sc.defaultParallelism, thời gian khởi tạo,
   và nhắc mở http://localhost:4040
```

### Công cụ

`SparkSession.builder` · `.master()` · `.config()` · `.getOrCreate()` · `sparkContext`


#### GỢI Ý GIẢI — D1

```python
from pyspark.sql import SparkSession, functions as F

t0 = time.perf_counter()
spark = (SparkSession.builder
         .appName("Buoi05_Spark")
         .master("local[*]")                           # mỗi lõi CPU là một "máy"
         .config("spark.sql.shuffle.partitions", "8")  # mặc định 200 — quá nhiều cho máy cá nhân
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
t_khoi_tao = time.perf_counter() - t0
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
print("Phiên bản Spark     :", spark.version)
print("Chế độ chạy         :", sc.master)
print("Số lõi khả dụng     :", sc.defaultParallelism)
print(f"Thời gian khởi tạo  : {t_khoi_tao:.1f} giây")
print("Spark UI            : http://localhost:4040")
```

**Bốn tầng công việc — thuộc bốn từ này là đọc được Spark UI:**

```text
JOB     — sinh ra bởi MỘT action (.collect(), .count(), .show(), .save())
 └─ STAGE  — đoạn công việc KHÔNG có shuffle bên trong.
    │        Ranh giới giữa hai stage LUÔN LUÔN là một lần shuffle.
    └─ TASK   — một stage chạy trên một phân vùng. 8 phân vùng = 8 task.
```

> **Vì sao đổi `spark.sql.shuffle.partitions` xuống 8?** Mặc định Spark tạo **200 phân vùng** sau mỗi shuffle — hợp lý cho cụm hàng trăm lõi, nhưng trên laptop nó tạo 200 task tí hon mà chi phí lập lịch còn lớn hơn công việc thật. Quy tắc thực dụng: **2–3 lần số lõi**.

> **Việc bắt buộc làm trên trình duyệt: mở http://localhost:4040 NGAY BÂY GIỜ.** Lúc này chưa có job nào. Từ D2 trở đi, mỗi action sẽ đẻ ra một dòng ở tab **Jobs** — đó là cách nhìn thấy lazy evaluation bằng mắt.
>
> **Lưu ý sống còn:** cổng 4040 chỉ tồn tại khi SparkSession còn sống. `spark.stop()` là trang đó chết ngay. Nếu 4040 bị chiếm, Spark tự nhảy sang 4041.

> **Chế độ `local[*]` nghĩa là gì?** Driver và Executor nằm **chung một tiến trình JVM**, `*` = dùng hết số lõi. Mọi khái niệm phân vùng, stage, shuffle đều **hoạt động thật**, chỉ là dữ liệu không đi qua mạng. Đây là chế độ mọi kỹ sư dữ liệu dùng khi phát triển.


In [ ]:
# --- D1: Khởi tạo SparkSession ---
# TODO ①: import SparkSession và functions as F
# TODO ②: Tạo SparkSession local[*] với shuffle.partitions=8, đo thời gian khởi tạo
# TODO ③: setLogLevel("ERROR") và gán sc = spark.sparkContext
# TODO ④: In phiên bản, master, defaultParallelism, thời gian, địa chỉ Spark UI

spark = None   # <-- gán SparkSession vào đây
sc    = None   # <-- gán spark.sparkContext vào đây


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D1 (không cần sửa)
# =============================================================================
def _kiem_tra_d1():
    if spark is None or sc is None:
        return print("[CHUA DAT] Chưa gán biến spark / sc. Hãy hoàn thành TODO ② và ③.")
    try:
        n = sc.parallelize(range(1000), 4).map(lambda x: x * 2).sum()
    except Exception as e:
        return print("[CHUA DAT] SparkContext không chạy được phép tính thử:", e)

    print(f"Phiên bản Spark      : {spark.version}")
    print(f"Chế độ chạy          : {sc.master}")
    print(f"Số lõi khả dụng      : {sc.defaultParallelism}")
    print(f"Phép tính thử (0..999 nhân 2) = {n:,}   (kỳ vọng 999.000)")
    print("-" * 62)
    if n == 999000 and sc.master.startswith("local"):
        print("[DAT] SparkSession sống và tính đúng.")
        print("      Mở http://localhost:4040 — tab Jobs vừa có thêm một dòng do phép tính thử này.")
    elif not sc.master.startswith("local"):
        print("[CHUA DAT] Bạn không dùng local[*]. Xem lại CẢNH BÁO ở đầu notebook.")

_kiem_tra_d1()


---
# D2. RDD API — WORDCOUNT
### 4 phút

### Nhiệm vụ

Viết lại **đúng thuật toán MapReduce của Buổi 04** bằng RDD API, và đối chiếu từng con số.

```text
BUỔI 04                                BUỔI 05 — RDD
hdfs dfs -cat corpus.txt         ──►   sc.textFile(...)
wc_mapper.py  tách dòng → từ     ──►   .flatMap(...)
wc_mapper.py  print(tu + "\t1")  ──►   .map(lambda tu: (tu, 1))
sort  (Shuffle & Sort)           ──►   ngầm bên trong reduceByKey
wc_reducer.py cộng dồn           ──►   .reduceByKey(lambda a, b: a + b)
```

### Kết quả mong đợi

```text
Số phân vùng    : 2
Số từ khác nhau : 373        ◄── TRÙNG KHỚP Buổi 04
Tổng số từ      : 1144       ◄── TRÙNG KHỚP Buổi 04
   một            43
   dữ             27
   liệu           27
   của            26
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Viết WordCount bằng RDD API của Spark, dùng sc, CORPUS_PATH, DAU_CAU, HA_ASCII
đã có sẵn trong ô thiết lập:

1. dong = sc.textFile(str(CORPUS_PATH)); in dong.getNumPartitions()
2. Xây chuỗi RDD, đo thời gian bằng time.perf_counter():
   - .flatMap(lambda d: d.translate(DAU_CAU).translate(HA_ASCII).split())
   - .map(lambda tu: (tu, 1))
   - .reduceByKey(lambda a, b: a + b)
3. In số từ khác nhau bằng .count()
4. In tổng số từ bằng .map(lambda x: x[1]).sum()
5. In 10 từ nhiều nhất bằng .takeOrdered(10, key=lambda x: (-x[1], x[0]))
6. Lưu kết quả vào biến wc_rdd (RDD) và ket_qua_wc (dict {tu: so_lan}),
   lưu thời gian vào biến t_d2
```

### Công cụ

`sc.textFile` · `.flatMap` · `.map` · `.reduceByKey` · `.takeOrdered` · `.getNumPartitions`


#### GỢI Ý GIẢI — D2

```python
t0 = time.perf_counter()

dong = sc.textFile(str(CORPUS_PATH))              # ① giống  hdfs dfs -cat
print("Số phân vùng:", dong.getNumPartitions())

wc_rdd = (dong
          .flatMap(lambda d: d.translate(DAU_CAU).translate(HA_ASCII).split())  # ② = mapper.py
          .map(lambda tu: (tu, 1))                                              # ③ phát <khóa, 1>
          .reduceByKey(lambda a, b: a + b))                                     # ④ = reducer.py

so_tu_khac_nhau = wc_rdd.count()        # ACTION đầu tiên — mọi thứ chạy ở đây
tong_so_tu      = wc_rdd.map(lambda x: x[1]).sum()
top10           = wc_rdd.takeOrdered(10, key=lambda x: (-x[1], x[0]))
t_d2 = time.perf_counter() - t0

ket_qua_wc = dict(wc_rdd.collect())     # chỉ 373 cặp nên collect() an toàn

print("Số từ khác nhau :", so_tu_khac_nhau, "  (Buổi 04: 373)")
print("Tổng số từ      :", tong_so_tu, "  (Buổi 04: 1144)")
for tu, sl in top10:
    print(f"   {tu:<12}{sl:>5}")
print(f"\nThời gian D2: {t_d2:.2f} giây   "
      f"(Buổi 04 mất {TIMINGS_B04['yarn_wordcount_s']:.1f} giây)")
```

> **`reduceByKey` và `groupByKey` — câu hỏi phỏng vấn kinh điển:**
>
> ```python
> rdd.reduceByKey(lambda a, b: a + b)      # ĐÚNG
> rdd.groupByKey().mapValues(sum)          # SAI — cùng kết quả, chậm hơn nhiều
> ```
>
> `reduceByKey` **cộng cục bộ trên từng phân vùng TRƯỚC** rồi mới shuffle — chỉ **373** cặp đi qua mạng.
> `groupByKey` **shuffle toàn bộ 1.144 cặp thô** rồi mới cộng.
>
> Phép cộng cục bộ đó chính là cái mà Buổi 04 gọi là **combiner** — Spark làm tự động, MapReduce bắt bạn khai báo.

> **Vì sao `sc.textFile()` cho 2 phân vùng dù tệp chỉ 7 KB?** Vì `defaultMinPartitions` của Spark là **2**. Đây là lời nhắc: **số phân vùng của Spark KHÔNG liên quan đến số khối HDFS** của Buổi 04. Bạn điều khiển nó bằng tham số thứ hai của `textFile`, hoặc `repartition()` / `coalesce()`.

> **Nhìn vào `localhost:4040` sau khi chạy ô này:** tab Jobs có **ba** job mới (một cho `count`, một cho `sum`, một cho `takeOrdered`) — mỗi action một job. Mỗi job có **hai** stage: trước và sau `reduceByKey`. Ranh giới đó chính là shuffle.


In [ ]:
# --- D2: RDD API — WordCount ---
# TODO ①: dong = sc.textFile(str(CORPUS_PATH)); in số phân vùng
# TODO ②: flatMap tách từ (dùng DAU_CAU, HA_ASCII) -> map thành (tu, 1) -> reduceByKey cộng
# TODO ③: In số từ khác nhau, tổng số từ, và 10 từ nhiều nhất
# TODO ④: So thời gian với TIMINGS_B04["yarn_wordcount_s"]

wc_rdd     = None   # <-- RDD kết quả
ket_qua_wc = None   # <-- dict {tu: so_lan}
t_d2       = None   # <-- thời gian chạy


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D2 (không cần sửa)
# =============================================================================
def _kiem_tra_d2():
    if not isinstance(ket_qua_wc, dict):
        return print("[CHUA DAT] Chưa gán biến ket_qua_wc.")

    n_kn  = len(ket_qua_wc)
    n_tot = sum(ket_qua_wc.values())
    print(f"Số từ khác nhau : {n_kn:>6,}   (Buổi 04: {CHUAN_WC['so_tu_khac_nhau']})")
    print(f"Tổng số từ      : {n_tot:>6,}   (Buổi 04: {CHUAN_WC['tong_so_tu']})")
    print("-" * 56)
    dat = (n_kn == CHUAN_WC["so_tu_khac_nhau"]) and (n_tot == CHUAN_WC["tong_so_tu"])
    for tu, sl in CHUAN_WC["top"].items():
        ok = ket_qua_wc.get(tu, 0) == sl
        dat &= ok
        print(f"  {'[DAT]' if ok else '[CHUA DAT]'} '{tu}' = {ket_qua_wc.get(tu, 0)}   (kỳ vọng {sl})")
    print("-" * 56)

    if dat:
        print("\n[DAT] Bốn phép Spark cho ra ĐÚNG kết quả của ba chương trình MapReduce ở Buổi 04.")
        if isinstance(t_d2, float):
            t_mr = TIMINGS_B04["yarn_wordcount_s"]
            print(f"      Spark {t_d2:.2f} s  ·  MapReduce {t_mr:.1f} s  ->  nhanh hơn {t_mr/t_d2:.0f} lần")
            print("      (nhưng hãy giữ nghi ngờ về con số này — D7 sẽ chỉ ra vì sao nó chưa công bằng)")
    else:
        print("\n[CHUA DAT] Nguyên nhân gần như chắc chắn: quy tắc chuẩn hóa văn bản khác Buổi 04.")
        print("           Phải dùng ĐÚNG DAU_CAU và HA_ASCII trong ô thiết lập,")
        print("           KHÔNG dùng str.lower() (nó hạ cả Đ thành đ và làm gộp từ).")

_kiem_tra_d2()


---
# D3. DATAFRAME API — DOANH THU THEO NGÀNH HÀNG
### 5 phút · **TRỌNG TÂM BUỔI HỌC**

### Nhiệm vụ

Chạy lại bài toán D7 của Buổi 04 bằng DataFrame API.

### Kết quả mong đợi

```text
+----------+------------+-----------+-------+
|category  |so_giao_dich|doanh_thu  |tb_don |
+----------+------------+-----------+-------+
|Gia dụng  |32231       |58925123000|1828213|
|Điện tử   |36296       |57872658000|1594464|
|Thời trang|27778       |47277119000|1701963|
|Thực phẩm |59856       |18555159000|309997 |
|Mỹ phẩm   |19839       |17421590000|878149 |
|Sách      |24000       |6693153000 |278881 |
+----------+------------+-----------+-------+
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Tính doanh thu theo ngành hàng bằng DataFrame API, dùng spark, F, TX_PATH:

1. df = spark.read.csv(str(TX_PATH), header=True, inferSchema=True)
2. In df.printSchema(), số dòng df.count(), số phân vùng df.rdd.getNumPartitions()
3. Đo thời gian bằng time.perf_counter() cho chuỗi:
   .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
   .groupBy("category")
   .agg(F.count("*").alias("so_giao_dich"),
        F.sum("revenue").alias("doanh_thu"),
        F.round(F.avg("revenue")).cast("long").alias("tb_don"))
   .orderBy(F.desc("doanh_thu"))
   rồi .show(truncate=False)
4. Lưu DataFrame vào biến doanh_thu, lưu dict {category: (so_giao_dich, doanh_thu)}
   vào biến ket_qua_revenue, lưu thời gian vào t_d3
```

### Công cụ

`spark.read.csv` · `.withColumn` · `.groupBy().agg()` · `F.sum` · `F.avg` · `.orderBy`


#### GỢI Ý GIẢI — D3

```python
df = spark.read.csv(str(TX_PATH), header=True, inferSchema=True)
df.printSchema()
print("Số dòng     :", df.count())
print("Số phân vùng:", df.rdd.getNumPartitions())

t0 = time.perf_counter()
doanh_thu = (df
    .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("category")
    .agg(F.count("*").alias("so_giao_dich"),
         F.sum("revenue").alias("doanh_thu"),
         F.round(F.avg("revenue")).cast("long").alias("tb_don"))
    .orderBy(F.desc("doanh_thu")))
doanh_thu.show(truncate=False)
t_d3 = time.perf_counter() - t0

ket_qua_revenue = {r["category"]: (r["so_giao_dich"], r["doanh_thu"])
                   for r in doanh_thu.collect()}
print(f"Thời gian D3: {t_d3:.2f} giây   "
      f"(Buổi 04 mất {TIMINGS_B04['yarn_revenue_s']:.1f} giây)")
```

**Ba điều bắt buộc rút ra:**

1. **Sáu con số trùng khớp tuyệt đối với Buổi 04** — kể cả cột `tb_don` mà Buổi 04 phải tự viết trong reducer. Cùng dữ liệu, cùng phép tính, hai công nghệ khác hẳn nhau, một kết quả.

2. **Bốn dòng Python thay cho hai chương trình + một job.** Buổi 04 cần `tx_mapper.sh`, `tx_reducer.sh`, một lệnh `hadoop jar` sáu tham số, và một thư mục đầu ra phải xóa trước.

3. **`avg` làm được ngay.** Buổi 04 phải giữ **cả tổng lẫn số đếm** trong reducer rồi chia thủ công, vì `AVG` không cộng dồn được. Spark lo việc đó — nhưng bản chất bên dưới vẫn thế, `explain()` ở D5 sẽ cho thấy `partial_avg` xuất hiện **trước** shuffle.

> **Hai cái bẫy khi đọc CSV:**
>
> 1. **`inferSchema=True` đọc tệp HAI LẦN** — một lần đoán kiểu, một lần đọc thật. Với dữ liệu lớn hãy khai báo lược đồ tường minh bằng `StructType`, hoặc dùng Parquet (đã có lược đồ nhúng bên trong).
> 2. **Số phân vùng của CSV do Spark tự quyết** theo kích thước tệp và số lõi — **không liên quan** đến khối HDFS. Ở đây `transactions.csv` cho **5 phân vùng**, trong khi trên HDFS nó có **3 khối**.

> **Vì sao `df.count()` chậm hơn bạn tưởng?** Vì nó là một **action**: Spark đọc và phân tích lại toàn bộ CSV. Gọi `count()` ba lần là đọc tệp ba lần. Đây chính là vấn đề mà D6 giải bằng `cache()`.


In [ ]:
# --- D3: DataFrame API — Doanh thu theo ngành hàng ---
# TODO ①: Đọc TX_PATH bằng spark.read.csv(header=True, inferSchema=True)
# TODO ②: In printSchema(), số dòng, số phân vùng
# TODO ③: withColumn revenue -> groupBy("category") -> agg(count, sum, round(avg)) -> orderBy
# TODO ④: show() và lưu kết quả

df              = None   # <-- DataFrame giao dịch
doanh_thu       = None   # <-- DataFrame kết quả
ket_qua_revenue = None   # <-- {category: (so_giao_dich, doanh_thu)}
t_d3            = None   # <-- thời gian chạy


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D3 (không cần sửa)
# =============================================================================
def _kiem_tra_d3():
    if not isinstance(ket_qua_revenue, dict):
        return print("[CHUA DAT] Chưa gán biến ket_qua_revenue.")

    print(f"  {'category':<12}{'giao dịch':>11}{'doanh thu':>18}   Đối chiếu Buổi 04")
    print("  " + "-" * 62)
    dat = True
    for k, (n, r) in sorted(CHUAN_REVENUE.items(), key=lambda x: -x[1][1]):
        thuc = ket_qua_revenue.get(k)
        ok = (thuc == (n, r))
        dat &= ok
        hien = f"{thuc[0]:>11,}{thuc[1]:>18,}" if thuc else f"{'—':>11}{'—':>18}"
        print(f"  {k:<12}{hien}   {'[DAT] khớp' if ok else '[CHUA DAT] lệch'}")
    tong = sum(v[1] for v in ket_qua_revenue.values())
    print("  " + "-" * 62)
    print(f"  {'TỔNG':<12}{sum(v[0] for v in ket_qua_revenue.values()):>11,}{tong:>18,}")
    print(f"  (kỳ vọng 200.000 giao dịch · {TONG_DOANH_THU:,} VND)")

    if dat and tong == TONG_DOANH_THU:
        print("\n[DAT] Bốn dòng DataFrame cho ra đúng kết quả mà Buổi 04 cần hai chương trình")
        print("      awk, một lệnh hadoop jar sáu tham số và ~30 giây để tính ra.")
    else:
        print("\n[CHUA DAT] Kiểm tra lại: revenue = quantity * unit_price, gom nhóm theo 'category'.")

_kiem_tra_d3()


---
# D4. SPARK SQL — BA BÁO CÁO MỚI
### 5 phút

### Nhiệm vụ

Đăng ký DataFrame thành bảng tạm, rồi viết **đúng cú pháp SQL của Buổi 03** trên dữ liệu phân tán.

### Kết quả mong đợi

*Top 5 sản phẩm:* `Quạt điều hòa` 26.796.034.000 · `Bàn phím cơ` 18.430.372.000 · `Ổ cứng SSD 512GB` 16.763.176.000

*Theo tháng:* 2025-01 → 38.781.757.000 … 2025-06 → **15.522.980.000** (chỉ 15.096 giao dịch)

*Theo phương thức:* TIEN_MAT 69.446.993.000 · THE 62.621.625.000 · VI_DIEN_TU 53.813.087.000 · CHUYEN_KHOAN 20.863.097.000

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Dùng Spark SQL để viết ba báo cáo mới:

1. Đăng ký bảng tạm:
   df.withColumn("revenue", F.col("quantity")*F.col("unit_price"))
     .createOrReplaceTempView("tx")

2. spark.sql: top 5 sản phẩm theo SUM(revenue), GROUP BY product,
   ORDER BY doanh_thu DESC LIMIT 5

3. spark.sql: doanh thu theo tháng, dùng date_format(ts,'yyyy-MM') AS thang,
   COUNT(*) và SUM(revenue), GROUP BY thang ORDER BY thang

4. spark.sql: doanh thu theo payment_method, COUNT(*) và SUM(revenue),
   ORDER BY doanh_thu DESC

In cả ba bảng bằng .show(truncate=False).
Lưu kết quả top 5 sản phẩm vào biến top_san_pham dạng list các tuple (product, doanh_thu).
```

### Công cụ

`createOrReplaceTempView` · `spark.sql()` · `date_format` · `GROUP BY` · `ORDER BY ... LIMIT`


#### GỢI Ý GIẢI — D4

```python
df.withColumn("revenue", F.col("quantity") * F.col("unit_price")) \
  .createOrReplaceTempView("tx")

print("=== ① TOP 5 SẢN PHẨM THEO DOANH THU ===")
q1 = spark.sql("""
    SELECT product, SUM(revenue) AS doanh_thu
    FROM tx GROUP BY product ORDER BY doanh_thu DESC LIMIT 5
""")
q1.show(truncate=False)
top_san_pham = [(r["product"], r["doanh_thu"]) for r in q1.collect()]

print("=== ② DOANH THU THEO THÁNG ===")
spark.sql("""
    SELECT date_format(ts, 'yyyy-MM') AS thang,
           COUNT(*) AS so_giao_dich, SUM(revenue) AS doanh_thu
    FROM tx GROUP BY thang ORDER BY thang
""").show(truncate=False)

print("=== ③ DOANH THU THEO PHƯƠNG THỨC THANH TOÁN ===")
spark.sql("""
    SELECT payment_method, COUNT(*) AS so_giao_dich, SUM(revenue) AS doanh_thu
    FROM tx GROUP BY payment_method ORDER BY doanh_thu DESC
""").show(truncate=False)
```

**Ba điểm dạy từ ba bảng này:**

1. **Tháng 6 chỉ có 15.096 giao dịch, chưa bằng một nửa các tháng khác.** Đây **không** phải sụt giảm kinh doanh — bộ dữ liệu chỉ ghi tới **giữa tháng 6**. **Luôn kiểm tra biên của dữ liệu trước khi kết luận về xu hướng.** Đây là lỗi báo cáo phổ biến nhất trong thực tế.

2. **`ORDER BY ... LIMIT 5` là thứ MapReduce làm rất cực.** Buổi 04 (Phụ lục A, bài 2) đã chỉ ra: reducer chỉ nhìn thấy **một khóa tại một thời điểm** nên không xếp hạng toàn cục được — phải chạy **job thứ hai**. Spark làm trong một câu.

3. **Ba câu SQL này chạy được nguyên văn trên PostgreSQL của Buổi 03.** Cùng ngôn ngữ, khác động cơ: một bên chạy trên một máy chủ, một bên chạy phân tán trên cả cụm. Đó là giá trị lớn nhất của Spark SQL — **kỹ năng SQL của bạn chuyển thẳng sang Big Data**.

> **DataFrame API hay Spark SQL — chọn cái nào?** Cả hai **biên dịch về cùng một kế hoạch Catalyst**, hiệu năng **y hệt nhau**. Chọn theo người đọc: SQL dễ cho người làm nghiệp vụ, DataFrame API dễ kiểm thử và ghép thành hàm Python. Trộn cả hai trong một chương trình là hoàn toàn bình thường.


In [ ]:
# --- D4: Spark SQL — ba báo cáo mới ---
# TODO ①: createOrReplaceTempView("tx") sau khi thêm cột revenue
# TODO ②: Top 5 sản phẩm theo doanh thu
# TODO ③: Doanh thu theo tháng — dùng date_format(ts, 'yyyy-MM')
# TODO ④: Doanh thu theo phương thức thanh toán

top_san_pham = None   # <-- list các tuple (product, doanh_thu), 5 phần tử


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D4 (không cần sửa)
# =============================================================================
CHUAN_TOP_SP = [("Quạt điều hòa",    26796034000),
                ("Bàn phím cơ",      18430372000),
                ("Ổ cứng SSD 512GB", 16763176000),
                ("Túi xách",         15553674000),
                ("Nồi cơm điện",     13909505000)]

def _kiem_tra_d4():
    if not top_san_pham:
        return print("[CHUA DAT] Chưa gán biến top_san_pham.")

    print(f"  {'#':<3}{'product':<20}{'doanh thu':>18}   Đối chiếu")
    print("  " + "-" * 58)
    dat = True
    for i, (ten, dt) in enumerate(CHUAN_TOP_SP, 1):
        thuc = top_san_pham[i-1] if i <= len(top_san_pham) else None
        ok = thuc is not None and thuc[0] == ten and int(thuc[1]) == dt
        dat &= ok
        hien = f"{thuc[0]:<20}{int(thuc[1]):>18,}" if thuc else f"{'—':<20}{'—':>18}"
        print(f"  {i:<3}{hien}   {'[DAT]' if ok else '[CHUA DAT] kỳ vọng ' + ten}")
    print("  " + "-" * 58)

    if dat:
        print("\n[DAT] Bạn vừa dùng SQL của Buổi 03 trên động cơ phân tán của Buổi 05.")
        print("      Chú ý: 'ORDER BY ... LIMIT 5' là thứ MapReduce phải chạy JOB THỨ HAI mới làm được.")
        print("\n      Câu hỏi bắt buộc trả lời: vì sao tháng 2025-06 chỉ có 15.096 giao dịch?")
        print("      (Gợi ý: KHÔNG phải vì kinh doanh sụt giảm.)")
    else:
        print("\n[CHUA DAT] Kiểm tra: GROUP BY product, SUM(revenue), ORDER BY DESC, LIMIT 5.")

_kiem_tra_d4()


---
# D5. LAZY EVALUATION VÀ `explain()`
### 5 phút · **TRỌNG TÂM BUỔI HỌC**

### Nhiệm vụ

Chứng minh **bằng số** rằng transformation không tính gì, rồi đọc kế hoạch vật lý để **đếm shuffle**.

```text
NARROW (hẹp) — mỗi phân vùng vào sinh đúng một phân vùng ra. KHÔNG qua mạng. RẺ.
   map · filter · flatMap · select · withColumn · union · coalesce

WIDE (rộng) — một phân vùng ra cần dữ liệu từ NHIỀU phân vùng vào. PHẢI shuffle. ĐẮT.
   groupByKey · reduceByKey · join · distinct · orderBy · repartition
   ^^^ Là RANH GIỚI giữa hai STAGE. Trong explain() nó hiện ra với tên "Exchange".
```

### Kết quả mong đợi

```text
Dựng chuỗi 3 biến đổi :   3,2 ms      ◄── gần như bằng 0: CHƯA TÍNH GÌ
Gọi .collect()        : 480,5 ms      ◄── TẤT CẢ chạy ở đây

Kế hoạch NARROW: 0 dòng Exchange
Kế hoạch WIDE  : 2 dòng Exchange  (một do groupBy, một do orderBy)
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Chứng minh lazy evaluation và đọc kế hoạch thực thi của Spark:

1. Đo thời gian DỰNG một chuỗi 3 transformation (withColumn + filter + groupBy/agg)
   bằng time.perf_counter(). KHÔNG gọi action.
2. Đo thời gian gọi .collect() trên chuỗi đó. So sánh hai con số.
3. In kế hoạch NARROW: df.filter(F.col("quantity") > 3).select("category","quantity").explain()
4. In kế hoạch WIDE: doanh_thu.explain()
5. Bắt đầu ra của explain() bằng io.StringIO + contextlib.redirect_stdout.
6. Viết hàm dem_exchange(ke_hoach) đếm số dòng bắt đầu bằng "Exchange".
   LƯU Ý: nếu AQE đã chạy xong, explain() in RA HAI kế hoạch — "== Final Plan =="
   và "== Initial Plan ==" — nên phải cắt lấy phần sau "== Initial Plan =="
   trước khi đếm, nếu không con số bị nhân đôi.
7. Lưu số Exchange của kế hoạch wide vào biến so_exchange.
```

### Công cụ

`.explain()` · `contextlib.redirect_stdout` · `time.perf_counter()`


#### GỢI Ý GIẢI — D5

```python
import contextlib, io

# ① Transformation KHÔNG tính gì
t0 = time.perf_counter()
tam = (df.withColumn("revenue", F.col("quantity") * F.col("unit_price"))
         .filter(F.col("quantity") >= 3)
         .groupBy("city").agg(F.sum("revenue").alias("rev")))
t_dung = (time.perf_counter() - t0) * 1000

# ② Action mới kích hoạt
t0 = time.perf_counter()
tam.collect()
t_action = (time.perf_counter() - t0) * 1000

print(f"Dựng chuỗi 3 biến đổi : {t_dung:8.1f} ms   <-- CHƯA TÍNH GÌ")
print(f"Gọi .collect()        : {t_action:8.1f} ms   <-- TẤT CẢ chạy ở đây")
print(f"Chênh nhau {t_action/t_dung:,.0f} lần")

# ③④ Bắt đầu ra của explain() để đếm Exchange
def ke_hoach(d):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        d.explain()
    return buf.getvalue()

def dem_exchange(kh):
    # Nếu AQE đã chạy xong, explain() in RA HAI kế hoạch (Final Plan + Initial Plan).
    # Chỉ đếm phần Initial Plan để con số không bị nhân đôi.
    if "== Initial Plan ==" in kh:
        kh = kh.split("== Initial Plan ==", 1)[1]
    return sum(1 for d in kh.split("\n") if d.strip().removeprefix("+- ").startswith("Exchange"))

kh_narrow = ke_hoach(df.filter(F.col("quantity") > 3).select("category", "quantity"))
kh_wide   = ke_hoach(doanh_thu)

print("\n=== KẾ HOẠCH NARROW (không shuffle) ===")
print(kh_narrow)
print("=== KẾ HOẠCH WIDE (có shuffle) ===")
print(kh_wide)

so_exchange = dem_exchange(kh_wide)
print(f"Số Exchange — narrow: {dem_exchange(kh_narrow)}   wide: {so_exchange}")
```

> **Một chi tiết thật của Spark 3+ mà bạn sẽ đụng ngay ở ô này.** Nếu DataFrame đã từng chạy một action, `explain()` in ra **hai** kế hoạch:
>
> - `== Final Plan ==` — kế hoạch **sau khi** AQE (*Adaptive Query Execution*) chỉnh lại dựa trên kích thước dữ liệu **thật** đo được lúc chạy. Bạn sẽ thấy thêm `AQEShuffleRead coalesced`: AQE đã **gộp các phân vùng nhỏ** sau shuffle.
> - `== Initial Plan ==` — kế hoạch Catalyst dựng **trước khi** chạy.
>
> Đếm `Exchange` trên cả hai sẽ ra **4** thay vì **2**. Đây không phải lỗi của bạn — đó là lý do hàm `dem_exchange` phải cắt lấy phần `Initial Plan`.
>
> Và bản thân `AQEShuffleRead coalesced` là một điểm dạy: Spark 3 **tự sửa** con số `spark.sql.shuffle.partitions` mà bạn đặt ở D1, dựa trên dữ liệu thật.

**Bốn điều bắt buộc rút ra — phần đắt giá nhất của buổi học:**

1. **Ba biến đổi mất vài mili giây, một action mất hàng trăm mili giây.** Lazy evaluation không phải lý thuyết — nó **đo được**.

2. **`PushedFilters` trong kế hoạch narrow là Catalyst đang làm việc cho bạn.** Bạn viết "đọc tệp rồi lọc", Catalyst đổi thành "lọc **ngay lúc đọc**". Và `ReadSchema` chỉ có 2 cột dù tệp có 10 — nó **không đọc** 8 cột kia. Hai kỹ thuật này tên là *predicate pushdown* và *column pruning*.

3. **Đếm `Exchange` là đếm shuffle.** Narrow có **0**, wide có **2** — một do `groupBy`, một do `orderBy`. Hai `Exchange` = **ba stage**. Mở tab **Stages** ở `localhost:4040` để nhìn thấy đúng ba khối đó.

4. **`partial_count` / `partial_sum` xuất hiện TRƯỚC `Exchange`** — đây chính là **combiner** của Buổi 04! Spark cộng cục bộ trên từng phân vùng trước, rồi mới gửi kết quả cục bộ qua mạng. Cùng ý tưởng với `reduceByKey` ở D2, và cùng ý tưởng combiner mà MapReduce bắt bạn tự khai báo.

> **Kỹ năng cần rèn suốt phần đời còn lại làm Spark:** đọc `explain()` **trước khi** chạy job lớn. Một `Exchange` thừa trên 1 TB dữ liệu là hàng chục phút và hàng trăm nghìn đồng tiền cụm. Nhìn kế hoạch mất 5 giây; chạy sai mất cả buổi chiều.


In [ ]:
# --- D5: Lazy evaluation và explain() ---
# TODO ①: Đo thời gian DỰNG một chuỗi 3 transformation (KHÔNG gọi action)
# TODO ②: Đo thời gian gọi .collect() trên chuỗi đó, so sánh
# TODO ③: In kế hoạch NARROW: df.filter(...).select(...).explain()
# TODO ④: In kế hoạch WIDE: doanh_thu.explain()
# TODO ⑤: Viết hàm dem_exchange() đếm số dòng bắt đầu bằng "Exchange".
#          CHÚ Ý: nếu có "== Initial Plan ==" thì chỉ đếm phần SAU nó,
#          nếu không con số sẽ bị nhân đôi (xem GỢI Ý GIẢI).

so_exchange = None   # <-- số lần shuffle trong kế hoạch wide


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D5 (không cần sửa)
# =============================================================================
def _kiem_tra_d5():
    import contextlib, io

    def ke_hoach(d):
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            d.explain()
        return buf.getvalue()

    def _dem(kh):
        if "== Initial Plan ==" in kh:
            kh = kh.split("== Initial Plan ==", 1)[1]
        return sum(1 for d in kh.split("\n") if d.strip().removeprefix("+- ").startswith("Exchange"))

    kh_n = ke_hoach(df.filter(F.col("quantity") > 3).select("category", "quantity"))
    kh_w = ke_hoach(doanh_thu)
    n_ex, w_ex = _dem(kh_n), _dem(kh_w)

    muc = [
        ("Kế hoạch NARROW không có Exchange nào",             n_ex == 0),
        ("Kế hoạch NARROW có PushedFilters (predicate pushdown)", "PushedFilters" in kh_n and "GreaterThan" in kh_n),
        ("Kế hoạch NARROW chỉ đọc 2 cột (column pruning)",    kh_n.count("category") >= 1 and "unit_price" not in kh_n),
        ("Kế hoạch WIDE có đúng 2 Exchange (groupBy + orderBy)", w_ex == 2),
        ("Kế hoạch WIDE có partial_* — chính là COMBINER của Buổi 04", "partial_" in kh_w),
        ("Bạn đã gán biến so_exchange đúng bằng số Exchange",  so_exchange == w_ex),
        ("Kế hoạch WIDE có AQEShuffleRead — AQE đã tự gộp phân vùng nhỏ",
         "AQEShuffleRead" in kh_w or "isFinalPlan=false" in kh_w),
    ]
    for ten, ok in muc:
        print(f"  {'[DAT] ' if ok else '[    ]'}  {ten}")
    print("-" * 68)
    print(f"  Narrow: {n_ex} Exchange   |   Wide: {w_ex} Exchange  ->  {w_ex + 1} stage")

    if all(ok for _, ok in muc):
        print("\n[DAT] Bạn đọc được kế hoạch thực thi của Spark. Đây là kỹ năng phân biệt")
        print("      người dùng Spark với người GỠ LỖI được Spark.")
        print("      Mở http://localhost:4040 -> tab Stages và đếm số stage của job vừa rồi.")
    else:
        print("\n[CHUA DAT] Còn mục trống. Hai chú ý:")
        print("           1. Phải explain() ĐÚNG hai DataFrame:")
        print("              narrow = df.filter(...).select(...)   ·   wide = doanh_thu")
        print("           2. Nếu đếm ra 4 Exchange thay vì 2: bạn đang đếm cả")
        print("              '== Final Plan ==' lẫn '== Initial Plan =='. Cắt lấy phần Initial Plan.")

_kiem_tra_d5()


---
# D6. `cache()` — ĐO LỢI ÍCH BẰNG SỐ
### 4 phút

### Nhiệm vụ

Chạy một khối lượng công việc **lặp** hai lần — không cache và có cache — rồi so.

### Vì sao cần cache?

```python
df2 = df.filter(...).withColumn(...)
df2.count()   # đọc lại tệp gốc, tính lại TỪ ĐẦU
df2.count()   # ĐỌC LẠI TỆP GỐC, TÍNH LẠI TỪ ĐẦU MỘT LẦN NỮA
```

RDD/DataFrame **không tự nhớ kết quả**. Mỗi action chạy lại trọn lineage.

### Kết quả mong đợi

```text
KHÔNG cache                1.63 s   (   204 ms/vòng)
CÓ cache                   0.53 s   (    66 ms/vòng)
Nhanh hơn 3.1 lần
StorageLevel: Disk Memory Deserialized 1x Replicated
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Đo lợi ích của cache() trên một khối lượng công việc lặp:

1. Tạo co_so = df.withColumn("revenue", quantity*unit_price).filter(quantity >= 2)
2. Viết hàm do_thoi_gian(d, nhan) chạy VONG = 8 vòng lặp, mỗi vòng gọi
   d.groupBy("category").agg(F.sum("revenue"), F.count("*")).collect(),
   đo tổng thời gian và in ra cả thời gian trung bình mỗi vòng.
3. Chạy lần 1 KHÔNG cache -> lưu vào t_khong_cache
4. Gọi co_so.cache() rồi co_so.count()   (BẮT BUỘC có count, xem ghi chú)
5. Chạy lần 2 CÓ cache -> lưu vào t_co_cache
6. In tỷ lệ nhanh hơn và co_so.storageLevel
7. Gọi co_so.unpersist() để trả lại bộ nhớ
```

### Công cụ

`.cache()` · `.persist(StorageLevel...)` · `.unpersist()` · `.storageLevel`


#### GỢI Ý GIẢI — D6

```python
co_so = (df.withColumn("revenue", F.col("quantity") * F.col("unit_price"))
           .filter(F.col("quantity") >= 2))
VONG = 8

def do_thoi_gian(d, nhan):
    t0 = time.perf_counter()
    for _ in range(VONG):
        d.groupBy("category").agg(F.sum("revenue"), F.count("*")).collect()
    dt = time.perf_counter() - t0
    print(f"{nhan:<24}{dt:7.2f} s   ({dt/VONG*1000:6.0f} ms/vòng)")
    return dt

t_khong_cache = do_thoi_gian(co_so, "KHÔNG cache")

co_so.cache()
co_so.count()          # BẮT BUỘC: action đầu tiên mới thực sự nạp dữ liệu vào bộ nhớ
t_co_cache = do_thoi_gian(co_so, "CÓ cache")

print(f"\nNhanh hơn {t_khong_cache/t_co_cache:.1f} lần")
print("StorageLevel:", co_so.storageLevel)
co_so.unpersist()
```

| **StorageLevel** | **Nghĩa** | **Dùng khi** |
|:---|:---|:---|
| `MEMORY_ONLY` | Chỉ RAM; không đủ thì phân vùng thừa bị **bỏ và tính lại** | RDD nhỏ, tính lại rẻ |
| `MEMORY_AND_DISK` | Không đủ RAM thì tràn xuống đĩa | **Mặc định của DataFrame — chọn cái này** |
| `MEMORY_ONLY_SER` | Nén ở dạng chuỗi byte: ít RAM hơn, tốn CPU hơn | RAM chật |
| `DISK_ONLY` | Chỉ đĩa | Tính lại rất đắt mà RAM không còn |

> **Ba quy tắc dùng `cache()`:**
>
> 1. **Chỉ cache thứ được dùng LẠI ít nhất hai lần.** Cache thứ dùng một lần là **lãng phí thuần túy**.
> 2. **Cache SAU khi đã lọc và chọn cột**, đừng cache bảng thô. Cache 10 cột trong khi chỉ dùng 3 là ném đi 70% bộ nhớ.
> 3. **`unpersist()` khi xong.** RAM của executor là tài nguyên chung.

> **`cache()` cũng LƯỜI — đây là cái bẫy khi đo hiệu năng.** `cache()` chỉ *đánh dấu* "hãy giữ lại kết quả", chứ không tính gì. Phải có một **action** chạy qua thì dữ liệu mới thực sự vào bộ nhớ. Quên `count()` sau `cache()` thì vòng lặp đầu tiên gánh luôn chi phí nạp cache và số đo sẽ sai.

> **Vì sao "chỉ" khoảng 3 lần chứ không phải 100 lần?** Vì `transactions.csv` chỉ 18,5 MB và chuỗi biến đổi rất ngắn. Khoảng cách rộng ra khi dữ liệu lớn hơn, chuỗi biến đổi dài hơn, hoặc số vòng lặp nhiều hơn.
>
> **Hãy đổi `VONG = 8` thành `VONG = 30` và chạy lại** — đó là bằng chứng trực quan cho câu "Spark sinh ra vì thuật toán lặp".

> **Kiểm chứng bằng mắt:** mở tab **Storage** ở `localhost:4040` **sau khi cache và trước khi `unpersist()`**. Bạn sẽ thấy đúng một mục với số phân vùng đã nạp và dung lượng RAM đang chiếm.


In [ ]:
# --- D6: cache() — đo lợi ích bằng số ---
# TODO ①: Tạo co_so = df + cột revenue + filter(quantity >= 2)
# TODO ②: Viết hàm do_thoi_gian(d, nhan) chạy VONG=8 vòng groupBy/agg/collect
# TODO ③: Đo KHÔNG cache -> t_khong_cache
# TODO ④: cache() + count() rồi đo CÓ cache -> t_co_cache
# TODO ⑤: In tỷ lệ nhanh hơn, in storageLevel, rồi unpersist()

t_khong_cache = None
t_co_cache    = None


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D6 (không cần sửa)
# =============================================================================
def _kiem_tra_d6():
    if not (isinstance(t_khong_cache, float) and isinstance(t_co_cache, float)):
        return print("[CHUA DAT] Chưa gán t_khong_cache / t_co_cache.")

    ty_le = t_khong_cache / t_co_cache
    print(f"KHÔNG cache : {t_khong_cache:6.2f} s")
    print(f"CÓ cache    : {t_co_cache:6.2f} s")
    print(f"Tỷ lệ       : {ty_le:6.2f} lần")
    print("-" * 56)

    if ty_le >= 1.5:
        print("[DAT] cache() có tác dụng đo được.")
        print("      Nhớ ba quy tắc: chỉ cache thứ dùng LẠI >= 2 lần · cache SAU khi lọc/chọn cột")
        print("      · unpersist() khi xong.")
        print("\n      Hãy đổi VONG = 8 thành VONG = 30 rồi chạy lại — khoảng cách sẽ rộng ra.")
    elif ty_le >= 1.0:
        print("[CHUA DAT] Có nhanh hơn nhưng rất ít. Hai nguyên nhân thường gặp:")
        print("   1. Quên gọi .count() sau .cache() -> vòng đầu gánh luôn chi phí nạp cache")
        print("   2. Máy quá nhanh so với 18,5 MB dữ liệu -> tăng VONG lên 30")
    else:
        print("[CHUA DAT] Có cache lại CHẬM HƠN. Gần như chắc chắn thiếu .count() sau .cache().")

_kiem_tra_d6()


---
---
# D7. SO SÁNH HIỆU NĂNG — VÀ CÁCH SO SÁNH CHO **CÔNG BẰNG**
### 4 phút

### Nhiệm vụ

Đọc `lab4/outputs/lab4_timings.json`, lập bảng so sánh, và **chỉ ra chỗ so sánh đó chưa công bằng**.

### Kết quả mong đợi

```text
Phép tính                     Buổi 04 (MR)   Buổi 05 (Spark)   Nhanh hơn
--------------------------------------------------------------------------
WordCount (7 KB)                    29.2 s            1.62 s        18.0x
Doanh thu (18,5 MB)                 29.2 s            1.28 s        22.8x
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Lập bảng so sánh hiệu năng giữa Buổi 04 và Buổi 05:
1. Dùng TIMINGS_B04 đã nạp sẵn (yarn_wordcount_s, yarn_revenue_s)
   và t_d2, t_d3 đo được ở D2, D3.
2. In bảng 4 cột: Phép tính | Buổi 04 (MR) | Buổi 05 (Spark) | Nhanh hơn
   cho hai dòng: "WordCount (7 KB)" và "Doanh thu (18,5 MB)".
3. Lưu bảng vào biến bang_so_sanh dạng list các dict
   {"phep_tinh", "mapreduce_s", "spark_s", "nhanh_hon"}.
```

### Công cụ

`TIMINGS_B04` · `t_d2` · `t_d3`


#### GỢI Ý GIẢI — D7

```python
bang_so_sanh = []
print(f"{'Phép tính':<34}{'Buổi 04 (MR)':>14}{'Buổi 05 (Spark)':>17}{'Nhanh hơn':>12}")
print("-" * 77)
for ten, t_mr, t_sp in [("WordCount (7 KB)",    TIMINGS_B04["yarn_wordcount_s"], t_d2),
                        ("Doanh thu (18,5 MB)", TIMINGS_B04["yarn_revenue_s"],   t_d3)]:
    print(f"{ten:<34}{t_mr:>12.1f} s{t_sp:>15.2f} s{t_mr/t_sp:>10.1f}x")
    bang_so_sanh.append({"phep_tinh": ten, "mapreduce_s": round(t_mr, 2),
                         "spark_s": round(t_sp, 2), "nhanh_hon": round(t_mr/t_sp, 1)})
```

---

### DỪNG LẠI. BẢNG TRÊN **KHÔNG CÔNG BẰNG**

Và việc nhận ra điều đó **quan trọng hơn chính con số**. Hãy tự tìm ra ba chỗ khập khiễng trước khi đọc tiếp.

<br>

1. **Khác phần cứng.** MapReduce chạy trong container Docker; Spark chạy thẳng trên máy thật. Trên máy Apple Silicon, container Hadoop còn chạy qua **lớp giả lập amd64** — chậm hơn đáng kể trước khi tính bất cứ điều gì khác.

2. **Khác thứ được tính vào.** Thời gian MapReduce gồm cả xin tài nguyên YARN và khởi động JVM; thời gian Spark **không** gồm mấy giây khởi tạo `SparkSession` vì phiên đã sẵn sàng từ D1.

3. **Khác quy mô dữ liệu so với hạ tầng.** 18,5 MB là quá nhỏ để bất kỳ hệ phân tán nào thể hiện đúng bản chất.

> **Bài học nghề nghiệp lớn nhất của buổi học:** một con số hiệu năng chỉ có nghĩa khi **mọi thứ khác được giữ nguyên**. Bảng trên vẫn hữu ích — nó phản ánh trải nghiệm thực tế của người dùng — nhưng gọi nó là *"Spark nhanh hơn MapReduce 23 lần"* thì **sai về phương pháp**.
>
> Vì vậy mới có D8.


In [ ]:
# --- D7: So sánh hiệu năng ---
# TODO ①: In bảng 4 cột so sánh WordCount và Doanh thu giữa Buổi 04 và Buổi 05
# TODO ②: Lưu vào biến bang_so_sanh
# TODO ③: Tự trả lời: bảng này chưa công bằng ở những chỗ nào?

bang_so_sanh = None   # <-- list các dict {phep_tinh, mapreduce_s, spark_s, nhanh_hon}


#### Câu trả lời của bạn — D7

**Bảng so sánh trên chưa công bằng ở những chỗ nào? Nêu ít nhất ba điểm.**

*(Viết câu trả lời tại đây.)*

**Nếu bạn là người viết báo cáo cho ban giám đốc, bạn sẽ trình bày con số này như thế nào cho trung thực?**

*(Viết câu trả lời tại đây.)*


---
# D8. SO SÁNH **CÔNG BẰNG** — SPARK CHẠY TRONG CHÍNH CỤM DOCKER
### 4 phút

### Nhiệm vụ

Chạy đúng hai phép tính đó bằng Spark **bên trong container**, đọc dữ liệu **thẳng từ HDFS** — cùng phần cứng, cùng lớp giả lập, cùng nguồn dữ liệu với job MapReduce Buổi 04.

Script đã có sẵn: `lab5/spark_jobs/revenue_hdfs.py` — **hãy mở ra đọc trước khi chạy**.

### Kết quả mong đợi

```text
Khởi tạo SparkSession :   0.71 s
Doanh thu theo ngành  :   4.24 s   (Buổi 04 mất ~30 s)
WordCount             :   0.82 s   (Buổi 04 mất ~30 s)
  Gia dụng       32,231    58,925,123,000
  ...
  Số từ khác nhau: 373
Tổng thời gian spark-submit (kể cả khởi động JVM): khoảng 8,5 giây
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Chạy Spark BÊN TRONG container để so sánh công bằng với MapReduce Buổi 04:

1. Kiểm tra container bigdata-spark-master có đang chạy không (docker ps).
   Nếu không, in hướng dẫn "docker compose up -d spark-master spark-worker" rồi dừng.
2. Kiểm tra dữ liệu còn trên HDFS không:
   docker exec bigdata-hadoop-namenode hdfs dfs -ls /user/bigdata/lab4/input
3. docker cp lab5/spark_jobs/revenue_hdfs.py bigdata-spark-master:/tmp/
4. Chạy và đo tổng thời gian bằng time.perf_counter():
   docker exec bigdata-spark-master /opt/bitnami/spark/bin/spark-submit \
       --master "local[*]" /tmp/revenue_hdfs.py
   Lọc bỏ các dòng log (bắt đầu bằng 2 chữ số, hoặc chứa WARN/INFO) rồi in phần còn lại.
5. Lưu tổng thời gian vào biến t_d8_container.
```

### Công cụ

`docker cp` · `spark-submit` · `subprocess.run`


#### GỢI Ý GIẢI — D8

```python
import re

SPARK_CT = "bigdata-spark-master"

# ① Container có sống không?
_ps = subprocess.run(["docker", "ps", "--format", "{{.Names}}"],
                     capture_output=True, text=True).stdout
if SPARK_CT not in _ps:
    print("Chưa thấy container", SPARK_CT, "— chạy:")
    print("   docker compose up -d hadoop-namenode hadoop-datanode spark-master spark-worker")
else:
    # ② Dữ liệu Buổi 04 còn trên HDFS không?
    ls = subprocess.run(["docker", "exec", "bigdata-hadoop-namenode",
                         "hdfs", "dfs", "-ls", "/user/bigdata/lab4/input"],
                        capture_output=True, text=True).stdout
    print("Dữ liệu trên HDFS:", "CÓ" if "transactions.csv" in ls else "KHÔNG — hãy chạy lại D2 của Buổi 04")

    # ③ Đưa script vào container
    subprocess.run(["docker", "cp", str(SPARK_JOBS / "revenue_hdfs.py"),
                    f"{SPARK_CT}:/tmp/"], check=True)

    # ④ Chạy spark-submit
    print("\nĐANG CHẠY SPARK TRONG CONTAINER (khoảng 10-20 giây)...")
    t0 = time.perf_counter()
    kq = subprocess.run(
        ["docker", "exec", SPARK_CT, "/opt/bitnami/spark/bin/spark-submit",
         "--master", "local[*]", "/tmp/revenue_hdfs.py"],
        capture_output=True, text=True)
    t_d8_container = time.perf_counter() - t0

    for d in (kq.stdout + kq.stderr).split("\n"):
        if d.strip() and not re.match(r"^\d{2}/", d) and "WARN" not in d and "INFO" not in d:
            print(d)
    print(f"\nTổng thời gian spark-submit (kể cả khởi động JVM): {t_d8_container:.1f} giây")
```

**BẢNG SO SÁNH CÔNG BẰNG — cùng máy, cùng container, cùng dữ liệu trên HDFS:**

| Phép tính | MapReduce trên YARN | Spark trong cụm | Nhanh hơn |
|:---|---:|---:|---:|
| WordCount (7 KB, từ HDFS) | ~29 s | **0,8 s** | **~36×** |
| Doanh thu (18,5 MB, từ HDFS) | ~29 s | **4,2 s** | **~7×** |
| Tổng cả hai, kể cả khởi động | ~58 s | **8,5 s** | **~7×** |

**Ba điều bắt buộc rút ra:**

1. **Kết quả giống hệt tới từng chữ số** — 373 từ, 206.744.802.000 VND. Spark **không tính khác**; nó chỉ **không ghi đĩa giữa các giai đoạn**.

2. **Hai phép tính chạy trong MỘT phiên Spark.** Buổi 04 cần **hai job Hadoop riêng biệt**, mỗi job trả lại chi phí khởi động từ đầu. Spark khởi động **một lần** rồi làm cả hai — càng nhiều bước liên tiếp, Spark càng thắng đậm.

3. **Chênh lệch không đồng đều: WordCount thắng ~36×, doanh thu chỉ thắng ~7×.** Vì WordCount xử lý 7 KB nên gần như toàn bộ 29 giây của MapReduce là **chi phí cố định** — bỏ được chi phí đó là thắng đậm. Còn với 18,5 MB, một phần thời gian là **tính toán thật**, mà tính toán thật thì Spark không làm biến mất được.

---

### KẾT LUẬN CẦN CHỐT CỦA CẢ BUỔI HỌC

> Spark nhanh hơn MapReduce **không phải vì thuật toán tốt hơn** — thuật toán y hệt: chia khóa, gom khóa, tổng hợp. Spark nhanh hơn vì **ba lý do rất cụ thể**:
>
> 1. **Không ghi dữ liệu trung gian xuống đĩa** giữa các giai đoạn.
> 2. **Một phiên làm nhiều việc**, thay vì mỗi job trả lại chi phí khởi động.
> 3. **Catalyst tối ưu lại kế hoạch** — đẩy bộ lọc xuống, cắt bớt cột, cộng cục bộ trước khi shuffle.
>
> Và Spark **không** thay thế Hadoop hoàn toàn: dữ liệu vẫn nằm trên **HDFS**, tài nguyên vẫn có thể do **YARN** cấp. Spark thay thế **MapReduce** — tầng tính toán — chứ không thay thế tầng lưu trữ. Đó là lý do hai buổi học này đi liền nhau.


In [ ]:
# --- D8: So sánh công bằng — Spark chạy trong container, đọc HDFS ---
# TODO ①: Kiểm tra container bigdata-spark-master có đang chạy không
# TODO ②: Kiểm tra dữ liệu Buổi 04 còn trên HDFS không
# TODO ③: docker cp lab5/spark_jobs/revenue_hdfs.py vào container
# TODO ④: Chạy spark-submit, lọc log, in kết quả, đo tổng thời gian

t_d8_container = None   # <-- tổng thời gian spark-submit (giây)


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D8 + GHI KẾT QUẢ BUỔI HỌC (không cần sửa)
# =============================================================================
def _kiem_tra_d8():
    if t_d8_container is None:
        print("[CHUA DAT] Chưa chạy được Spark trong container (bỏ qua được nếu máy không có Docker).")
        print("           D1-D7 vẫn tính đủ điểm; hãy trả lời câu hỏi về tính công bằng ở D7.")
        return False

    t_mr_tong = TIMINGS_B04["yarn_wordcount_s"] + TIMINGS_B04["yarn_revenue_s"]
    print(f"MapReduce — hai job trên YARN      : {t_mr_tong:6.1f} s")
    print(f"Spark     — một phiên trong cụm    : {t_d8_container:6.1f} s")
    print(f"Nhanh hơn (CÙNG môi trường)        : {t_mr_tong/t_d8_container:6.1f} lần")
    print("-" * 62)
    print("Đây mới là con số so sánh CÔNG BẰNG: cùng máy, cùng container,")
    print("cùng lớp giả lập, cùng dữ liệu đọc từ HDFS.")
    return True


_co_d8 = _kiem_tra_d8()

# --- Ghi kết quả buổi học ------------------------------------------------------
ket_qua_lab5 = {
    "spark_version":            spark.version if spark else None,
    "master":                   sc.master if sc else None,
    "so_loi":                   sc.defaultParallelism if sc else None,
    "t_d2_rdd_wordcount_s":     round(t_d2, 3) if isinstance(t_d2, float) else None,
    "t_d3_df_doanh_thu_s":      round(t_d3, 3) if isinstance(t_d3, float) else None,
    "t_d6_khong_cache_s":       round(t_khong_cache, 3) if isinstance(t_khong_cache, float) else None,
    "t_d6_co_cache_s":          round(t_co_cache, 3) if isinstance(t_co_cache, float) else None,
    "t_d8_spark_trong_cum_s":   round(t_d8_container, 2) if t_d8_container else None,
    "so_exchange_ke_hoach_wide": so_exchange,
    "timings_buoi_04":          TIMINGS_B04,
    "bang_so_sanh":             bang_so_sanh,
    "ket_qua_wordcount_top10":  sorted(ket_qua_wc.items(), key=lambda x: (-x[1], x[0]))[:10]
                                if isinstance(ket_qua_wc, dict) else None,
    "ket_qua_doanh_thu":        {k: list(v) for k, v in ket_qua_revenue.items()}
                                if isinstance(ket_qua_revenue, dict) else None,
}
(OUTPUT_DIR / "lab5_ket_qua.json").write_text(
    json.dumps(ket_qua_lab5, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nĐã ghi:", OUTPUT_DIR / "lab5_ket_qua.json")


---
---
# TỔNG KẾT BUỔI HỌC

## Sản phẩm phải nộp

| # | Tệp | Nội dung |
|:--|:---|:---|
| 1 | `lab5/Lab5.ipynb` | Notebook đã chạy hết, không còn ô TODO trống |
| 2 | `lab5/outputs/lab5_ket_qua.json` | Kết quả và thời gian của cả 8 bước |
| 3 | `lab5/outputs/report_lab05.md` | Báo cáo, kèm bảng so sánh công bằng và bình luận về tính công bằng |
| 4 | Ảnh chụp `localhost:4040` | Tab **Jobs**, tab **Stages** (thấy rõ ranh giới shuffle), tab **Storage** (sau khi cache) |
| 5 | Ảnh chụp kết quả `spark-submit` trong container | Bảng doanh thu 6 dòng và thời gian |

## Tiêu chí hoàn thành

- [ ] Tạo được SparkSession `local[*]`, mở được `http://localhost:4040`
- [ ] RDD WordCount ra đúng **373** từ khác nhau / **1.144** tổng số từ
- [ ] Giải thích được vì sao `reduceByKey` tốt hơn `groupByKey`
- [ ] DataFrame cho 6 ngành hàng khớp tuyệt đối, tổng **206.744.802.000** VND
- [ ] Viết được 3 truy vấn Spark SQL và đọc đúng ý nghĩa kết quả
- [ ] Chỉ ra được vì sao tháng 2025-06 chỉ có 15.096 giao dịch
- [ ] Chứng minh bằng số rằng transformation không tính gì, action mới tính
- [ ] Đếm đúng số `Exchange` và giải thích mỗi dòng do đâu
- [ ] Chỉ ra `partial_count`/`partial_sum` và nối được với **combiner** của Buổi 04
- [ ] Đo được lợi ích `cache()` và nêu đúng ba quy tắc dùng
- [ ] Lập bảng so sánh với Buổi 04 **và** chỉ ra ba chỗ chưa công bằng
- [ ] Chạy được Spark trong container đọc HDFS, có bảng so sánh công bằng

## Bảng chấm điểm

| Tiêu chí | Điểm |
|:---|---:|
| D1 — SparkSession đúng cấu hình, đọc được Spark UI | 10 |
| D2 — RDD WordCount khớp Buổi 04, giải thích `reduceByKey` vs `groupByKey` | 15 |
| D3 — DataFrame cho kết quả khớp tuyệt đối | 15 |
| D4 — Ba truy vấn Spark SQL đúng, đọc được ý nghĩa nghiệp vụ | 15 |
| D5 — Chứng minh lazy evaluation, đếm và giải thích `Exchange` | 20 |
| D6 — Đo được lợi ích `cache()`, nêu đúng quy tắc dùng | 10 |
| D7 — Bảng so sánh **và** phân tích tính công bằng | 10 |
| D8 — Chạy Spark trong cụm, bảng so sánh công bằng | 5 |
| **Tổng** | **100** |

---

## Lỗi thường gặp và cách xử lý

| Lỗi | Nguyên nhân | Cách xử lý |
|:---|:---|:---|
| `JAVA_HOME is not set` / `Java gateway process exited` | Chưa cài Java hoặc sai phiên bản | Cài JDK 17; PySpark 4.x **bắt buộc** Java ≥ 17 |
| `Python worker exited unexpectedly` | Python driver ≠ Python worker | Đặt `PYSPARK_PYTHON=sys.executable` |
| `Initial job has not accepted any resources` (treo mãi) | Nối vào cụm standalone thiếu tài nguyên hoặc **lệch phiên bản** | Dùng `.master("local[*]")` |
| `java.io.InvalidClassException` | PySpark trên máy ≠ Spark trong container | Cài đúng `pyspark==3.5.1`, hoặc dùng `local[*]` |
| `Port 4040 already in use` | Còn SparkSession khác đang sống | Không phải lỗi — Spark nhảy sang 4041 |
| Chạy ô lần hai vẫn rất chậm | Không cache, mỗi action đọc lại tệp | `.cache()` **+ một** `.count()` |
| `OutOfMemoryError` ở driver | `collect()` kéo **toàn bộ** dữ liệu về driver | Dùng `show(20)`, `take(n)`, hoặc `write` ra tệp |
| `AnalysisException: cannot resolve` | Sai tên cột, hoặc chưa `createOrReplaceTempView` | `df.printSchema()` trước khi viết truy vấn |
| WordCount lệch với Buổi 04 | Quy tắc chuẩn hóa khác | Dùng đúng `DAU_CAU` / `HA_ASCII` trong ô thiết lập |
| 200 task tí hon cho một `groupBy` | `spark.sql.shuffle.partitions` mặc định 200 | Đặt bằng 2–3 lần số lõi |
| Cảnh báo `NativeCodeLoader` | Thiếu thư viện native của Hadoop | **Không phải lỗi** — bỏ qua |

> **Mẹo gỡ lỗi quan trọng nhất:** khi một job Spark chạy lâu bất thường, đừng đoán — **mở `localhost:4040` → tab Stages → tìm stage có số task nhiều nhất hoặc thời gian lệch nhau nhất giữa các task**. Task lệch nhau lớn nghĩa là **data skew**: một khóa chiếm phần lớn dữ liệu, đúng hiện tượng đã gặp ở bài tập 4 của Buổi 04.

---

## Buổi tiếp theo

| Buổi | Chủ đề | Liên hệ với hôm nay |
|:---|:---|:---|
| **06** | Data Wrangling | Làm sạch dữ liệu bằng chính DataFrame API hôm nay, ở quy mô phân tán |
| **07** | Feature Engineering | `withColumn` và `CASE WHEN` thành bước tạo đặc trưng cho mô hình |
| **08+** | Spark Streaming / Kafka | Structured Streaming dùng **y hệt** DataFrame API — chỉ khác nguồn là luồng thay vì tệp |
| **13** | Học máy | Spark MLlib chạy trên DataFrame; `cache()` của D6 là **bắt buộc** vì thuật toán học máy lặp hàng chục vòng |

> Hôm nay `cache()` chỉ giúp nhanh hơn ~3 lần trên 8 vòng lặp. Buổi 13 sẽ chạy thuật toán lặp **hàng chục vòng** trên dữ liệu lớn hơn — ở đó, quên `cache()` là chênh nhau **hàng chục lần**.


In [ ]:
# =============================================================================
# NGHIỆM THU CUỐI BUỔI — chạy trước khi nộp bài (không cần sửa)
# =============================================================================
print("=" * 74)
print("BẢNG NGHIỆM THU BUỔI 05".center(74))
print("=" * 74)

_g   = globals()
_muc = []

_muc.append(("D1  SparkSession sống, chạy ở chế độ local",
             spark is not None and sc is not None and str(sc.master).startswith("local")))

_wc = _g.get("ket_qua_wc")
_muc.append(("D2  RDD WordCount: 373 từ khác nhau / 1.144 tổng số từ",
             isinstance(_wc, dict) and len(_wc) == 373 and sum(_wc.values()) == 1144))

_rv = _g.get("ket_qua_revenue")
_muc.append(("D3  DataFrame: 6 ngành hàng khớp tuyệt đối Buổi 04",
             isinstance(_rv, dict) and _rv == CHUAN_REVENUE))
_muc.append((f"D3  Tổng doanh thu = {TONG_DOANH_THU:,} VND",
             isinstance(_rv, dict) and sum(v[1] for v in _rv.values()) == TONG_DOANH_THU))

_tsp = _g.get("top_san_pham")
_muc.append(("D4  Spark SQL: top sản phẩm là 'Quạt điều hòa'",
             bool(_tsp) and _tsp[0][0] == "Quạt điều hòa"))

_muc.append(("D5  Đếm đúng 2 Exchange trong kế hoạch wide (groupBy + orderBy)",
             _g.get("so_exchange") == 2))

_a, _b = _g.get("t_khong_cache"), _g.get("t_co_cache")
_muc.append(("D6  cache() đo được lợi ích (nhanh hơn >= 1,5 lần)",
             isinstance(_a, float) and isinstance(_b, float) and _a / _b >= 1.5))

_muc.append(("D7  Đã lập bảng so sánh với Buổi 04",
             isinstance(_g.get("bang_so_sanh"), list) and len(_g.get("bang_so_sanh") or []) >= 2))

_muc.append(("D8  Đã chạy Spark trong cụm Docker (tùy chọn)",
             _g.get("t_d8_container") is not None))

_muc.append(("    Đã ghi outputs/lab5_ket_qua.json",
             (OUTPUT_DIR / "lab5_ket_qua.json").exists()))

_dat = 0
for _ten, _ok in _muc:
    print(f"  {'[DAT] ' if _ok else '[    ]'}  {_ten}")
    _dat += bool(_ok)

print("-" * 74)
print(f"HOÀN THÀNH: {_dat}/{len(_muc)} mục  ({_dat/len(_muc)*100:.0f}%)")
print("=" * 74)
if _dat >= len(_muc) - 1:
    print("Chúc mừng! Bạn đã hoàn thành trọn vẹn Buổi 05.")
    print("Bạn đã chạy cùng một phép tính bằng ba API của Spark — RDD, DataFrame, SQL —")
    print("và chứng minh được rằng Spark cho ra ĐÚNG kết quả của MapReduce,")
    print("nhanh hơn nhiều lần, mà không hề tính theo cách khác.")
    print("\nQuan trọng hơn cả con số: bạn biết vì sao một phép so sánh hiệu năng")
    print("có thể trông thuyết phục mà vẫn sai về phương pháp.")
else:
    print("Còn mục chưa xong (ô trống [    ]). Hoàn thiện rồi chạy lại ô này.")
print("=" * 74)

# Giải phóng tài nguyên. LƯU Ý: sau lệnh này, http://localhost:4040 sẽ tắt.
# Hãy chụp ảnh màn hình Spark UI TRƯỚC khi chạy ô này.
# spark.stop()
